# Tutorial: Gillespie algorithm

In [1]:
#user = r"\SagixOffice"  # HomeOffice
user = r"\vie43sq"  # University
import sys
sys.path.append(rf"C:\Users{user}\OneDrive - Universität Würzburg\GitHub\Photoswitching")

import numpy as np
import matplotlib.pyplot as plt
import src.fluorophore_systems as fs
import src.custom_plot as cp
import src.figures as fi
import src.gillespie_algorithm as ga
from IPython.display import HTML

%load_ext autoreload
%autoreload 2

In [2]:
rate_dict = dict(k_S0_S1=[7e6, "excitation"],  
                 k_S1_S0=[1e9, "fluorescent emission"], 
                 k_S1_T1=[1e6, "intersystem crossing"],   
                 k_T1_S0=[5e5, "vibrational relaxation"])

In [3]:
system = fs.GeneralModel(number=2, distances=1, rates=rate_dict)

In [4]:
def direct_method_py(row_sums, initial_row_vector, transition_matrix, rate_id_dict, n_steps, seed):
    rng = np.random.default_rng(seed)

    current_state = initial_row_vector
    print("The first current state is defined as ", initial_row_vector)

    time_step_series = np.empty(n_steps + 1)
    time_step_series[0] = 0
    state_series = np.empty(n_steps + 1, dtype=np.int64)

    transition_series = np.empty(n_steps, dtype=np.int64)

    current_state_index = state_series[0] = np.where(current_state == 1)[0][0]
    print("The first occupied state is at index ", current_state_index)
    random_numbers = rng.uniform(low=0, high=1, size=(n_steps, 2))
    print("For each step, 2 random numbers between 0 and 1 are generated")

    # the following block is to generate a sorted transition matrix to later check at which index the randomly drawn
    # number should be inserted
    print("The transition matrix has [number of states] rows and [number of states] columns." +
          " The current state index corresponds to the row index. Possible future states are in" +
          " the columns of that row. It contains the point probabilities for the corresponding transition" +
          " (e.g., the transition from state 0 to state 1 has a point probability of 0.5, then one can find" +
          " 0.5 as an entry in transition_matrix[0, 1])")
    print("The first row looks like this:", transition_matrix[0])
    transition_matrix_sorted_indices = np.argsort(transition_matrix, axis=1)
    sorted_transition_matrix = np.take_along_axis(transition_matrix, transition_matrix_sorted_indices, axis=1)
    print("The transition matrix is then being sorted from low to high values." + 
          " The first row then looks like this:", sorted_transition_matrix[0])
    print("The original indices are saved in new order:", transition_matrix_sorted_indices[0])
    cumsum_sorted_trm = np.cumsum(sorted_transition_matrix, axis=1)
    print("The cumulative sum of the sorted matrix is taken:", cumsum_sorted_trm[0])
    print("The last entry always has the value 1, if the state is not an absorbing state.")
    print("")
    future_state = None
    for i in range(n_steps):
        if i > 0:
            current_state_index = future_state
        current_state_lambda = row_sums[current_state_index]

        if current_state_lambda == 0:  # there is no possible transition going from the current state
            time_step_series = time_step_series[:i + 1]
            state_series = state_series[:i + 1]
            break

        transition_time = 1 / current_state_lambda * np.log(1 / random_numbers[i, 0])  # resembles inverse transform
        # sampling using the quantile function of the exponential distribution
        # the exponential distribution is used since it is the time between events in a Poisson point process
        # meaning that events occur continuously and independently at a constant average rate

        sorted_index = np.searchsorted(cumsum_sorted_trm[current_state_index], random_numbers[i, 1])  # get the
        # index of the sorted value list
        future_state = transition_matrix_sorted_indices[current_state_index, sorted_index]  # use the previous index to
        # get the original index of the sorted value

        identification = rate_id_dict[(current_state_index, future_state)]
        transition_series[i] = identification

        state_series[i + 1] = future_state
        time_step_series[i + 1] = transition_time
        if i == 0:
            print("A simulation step consists of the following:")
            print("First, the total rate (which, in this case, equals the total propensity) of the current state" +
                  " is taken (see row_sums):", current_state_lambda, "s\N{SUPERSCRIPT MINUS}\N{SUPERSCRIPT ONE}")
            print("Then, the transition time is calculated using the total rate (i.e., lambda) in the exponential" +
                  " distribution. The random variable is defined via inverse transform sampling using the first of" +
                  " the two random numbers of the step:", transition_time, "s")
            print("To determine, which of the possible future states is occupied after transition time passes," +
                  " the second of the two random numbers of the step is orderly inserted into the cumulative sum." +
                  " (of that state). I.e., if the array would be [0, 0, 0.1, 0.5, 1] and the random number is 0.2" + 
                  " it would be inserted at the previous index of 0.5, hence the future state will be the index" +
                  " corresponding to the cumulative sum of 0.5, which has been saved previously.")
            print("In the next step, the current occupied state is changed to the last future state.")
    time_series = np.cumsum(time_step_series)

    return time_series, time_step_series, state_series, transition_series

In [6]:
time_series, time_step_series, state_series, transition_series = \
    direct_method_py(system.row_sums, system.initial_row_vector, system.transition_matrix, 
                     system.rate_id_dict, n_steps=100, seed=100)

The first current state is defined as  [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
The first occupied state is at index  0
For each step, 2 random numbers between 0 and 1 are generated
The transition matrix has [number of states] rows and [number of states] columns. The current state index corresponds to the row index. Possible future states are in the columns of that row. It contains the point probabilities for the corresponding transition (e.g., the transition from state 0 to state 1 has a point probability of 0.5, then one can find 0.5 as an entry in transition_matrix[0, 1])
The first row looks like this: [0.  0.5 0.  0.  0.5 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0. ]
The transition matrix is then being sorted from low to high values. The first row then looks like this: [0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.5 0.5]
The original indices are saved in new order: [ 0  2  3  5  6  7  8  9 10 11 12 13 14 15  1  4]
The cumulative sum of the sorted matrix is take

In [30]:
%%time
dic =  dict(zip([(0, 1),(0,0),(1,0),(1,1),(2,1),(2,0)], [4, 9, 1, 5, 9, 10]))

CPU times: total: 0 ns
Wall time: 0 ns


In [31]:
dic

{(0, 1): 4, (0, 0): 9, (1, 0): 1, (1, 1): 5, (2, 1): 9, (2, 0): 10}

In [23]:
import numpy as np

In [24]:
arr = np.array([[9, 4],[1, 5], [10,9]])

In [44]:
index_one = np.random.randint(0, 3, 5000000)
index_two = np.random.randint(0, 2, 5000000)

In [45]:
index_two

array([1, 1, 1, ..., 1, 0, 1])

In [46]:
%%time
for i in range(5000000):
    l = index_one[i]
    k = index_two[i]
    value = dic[(l, k)]

CPU times: total: 3.45 s
Wall time: 3.45 s


In [47]:
%%time
for i in range(5000000):
    l = index_one[i]
    k = index_two[i]
    value = arr[l, k]

CPU times: total: 2.69 s
Wall time: 2.68 s
